# Houston COVID-19 GIS — NSF EAGER #2028612

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sear-labs/houston-covid-gis/blob/main/notebooks/00_walkthrough.ipynb)

**A thin notebook: it imports the package and calls it, and holds no analysis logic**, so it cannot
drift from the code. To read the analysis, read `src/houston_covid_gis/`.

The original was an ArcGIS Pro project whose central data service is now dead. This rebuild needs no
Esri licence, and the core — tables and connectivity — needs no geospatial stack either.


## 1. Install

On Colab, install from the repository. Locally, `pip install -e ".[dev]"` once.

In [ ]:
# Colab: clone the repository, then install FROM that checkout.
#
# `pip install git+https://...` is not enough, and fails in a way that looks
# like a bug in the model. It installs the package but not `data/`, which is
# not part of the wheel - and the installed `paths.py` then derives the repo
# root relative to site-packages, so the first cell that loads data dies with a
# FileNotFoundError naming a directory inside the interpreter. Cloning keeps the
# code and the data together in the layout `paths.py` expects.
import os
import subprocess
import sys

REPO = "houston-covid-gis"

if "google.colab" in sys.modules:
    if os.path.basename(os.getcwd()) != REPO:
        if not os.path.isdir(REPO):
            subprocess.run(
                ["git", "clone", "-q",
                 "https://github.com/sear-labs/" + REPO + ".git"], check=True)
        os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."],
                   check=True)
    print("cloned and installed", REPO, "into", os.getcwd())
else:
    print("local environment - assuming `pip install -e \".[dev]\"` has been run")


## 2. Where the data came from, and what died

The original map carried 25 layers and **no local data** — every one was a remote ArcGIS Online
service. Two no longer exist, including the one holding every COVID count. That is why the `.mapx`
alone cannot be replayed, and why the surviving file geodatabase is what this repo is built from.


In [2]:
from houston_covid_gis import sources

print(sources.status_report())

LIVE (7):
  cdc_svi_2016_houston_tracts
  coh_neighborhood_services
  county_health_rankings_2019
  healthcare_facilities
  jhu_covid_us
  rate_of_asthma
  usa_counties

RETIRED (2) - the map cannot be replayed from the network:
  CITY_LIMITS_COVID
      held:      per-ZIP TotalConfirmedCases / ActiveCases / Recovered / Death, and the polygons the eight regional layers selected from
      status:    GONE - returns 'Invalid URL'; absent from the City of Houston org's ~1000 published services
      recovered: data/derived/covid_by_region_zip.csv (attributes) and data/geometry/covid_regions.gpkg (geometry)
  Harris_Vulnerability_Census_Tracts
      held:      a Final_Vuln composite vulnerability score per census tract
      status:    GONE - returns 'Invalid URL'
      recovered: NOT RECOVERED. Substitute CDC SVI 2018 (see PUBLIC below), which is the input it was derived from, not the layer itself.


## 3. The regional picture

147 ZIP codes partitioned across eight regions — no overlap, no gaps. Generated from the
geodatabase, so the table and the geometry cannot disagree.


In [3]:
from houston_covid_gis import tables

summary = tables.regional_summary()
print(summary.to_string())
print()
print("total cases:  {:,}".format(summary["TotalConfirmedCases"].sum()))
print("total deaths: {:,}".format(summary["Death"].sum()))

        TotalConfirmedCases  ActiveCases  Recovered  Death  ZIPs  CaseFatalityPct
Region                                                                           
W                     10341         6908       3363     70    22             0.68
N                     10336         6194       4038    104    35             1.01
SW                     8306         6284       1932     90    18             1.08
S                      6536         5343       1132     61    17             0.93
E                      4396         2430       1908     58    16             1.32
SE                     3678         2444       1207     27    15             0.73
NW                     3216         1423       1763     30    11             0.93
NE                     2556         1248       1261     47    13             1.84

total cases:  49,365
total deaths: 487


Case fatality varies almost three-fold across the city. That spread is the health-equity signal
the project existed to find.


## 4. The exported spreadsheets disagree — kept for provenance, not used as data

The project also holds eight `*_COVID_TableToExcel.xlsx` exports. They are an earlier, partial
snapshot and they do **not** agree with the geodatabase. The disagreement is in the region
assignment, not just the totals.


In [4]:
legacy = tables.legacy_region_exports()
auth = tables.covid_by_region_zip()

print("geodatabase (authoritative): {:3} ZIPs, {:,} cases".format(
    len(auth), auth["TotalConfirmedCases"].sum()))
print("legacy xlsx exports:         {:3} ZIPs, {:,} cases".format(
    len(legacy), legacy["TotalConfirmedCases"].sum()))
print()
for r in ("W", "S", "SE"):
    a = set(auth[auth.Region == r].ZIP.astype(str))
    b = set(legacy[legacy.Region == r].ZIP.astype(str))
    print("  region {:2}: geodatabase {:2} ZIPs, legacy {:2} ZIPs, shared {:2}".format(
        r, len(a), len(b), len(a & b)))

geodatabase (authoritative): 147 ZIPs, 49,365 cases
legacy xlsx exports:         116 ZIPs, 13,954 cases

  region W : geodatabase 22 ZIPs, legacy  7 ZIPs, shared  2
  region S : geodatabase 17 ZIPs, legacy  2 ZIPs, shared  0
  region SE: geodatabase 15 ZIPs, legacy  2 ZIPs, shared  0


## 5. Connectivity — ArcGIS Network Analyst was never needed

`Generate Near Table` does no routing despite the branding: it is planar Euclidean. So the complete
88×88 matrix is ordinary graph work.

A percolation sweep the original could not produce — raise the link threshold and watch Houston's 88
super neighborhoods coalesce into one connected system.


In [5]:
from houston_covid_gis import connectivity

g = connectivity.distance_graph()
print("complete graph: {} nodes, {} edges".format(g.number_of_nodes(), g.number_of_edges()))
print("88 * 87 / 2 = {}, so the Near Table is complete".format(88 * 87 // 2))
print()
print(connectivity.percolation_sweep(10).to_string(index=False))

complete graph: 88 nodes, 3828 edges
88 * 87 / 2 = 3828, so the Near Table is complete



 threshold_ft  threshold_mi  components  largest_component  isolated
       5517.8          1.05          87                  2        86
      23948.6          4.54           7                 82         6
      42379.3          8.03           2                 86         0
      60810.1         11.52           1                 88         0
      79240.8         15.01           1                 88         0
      97671.6         18.50           1                 88         0
     116102.3         21.99           1                 88         0
     134533.1         25.48           1                 88         0
     152963.8         28.97           1                 88         0
     171394.6         32.46           1                 88         0
     189825.3         35.95           1                 88         0


## 6. Geometry — optional, needs the geospatial stack

Everything above runs on pandas and networkx alone. Reading the GeoPackage needs geopandas; if it is
absent this cell says so and the notebook still completes.


In [6]:
try:
    import geopandas as gpd
    from houston_covid_gis.paths import GEOMETRY

    print(gpd.list_layers(GEOMETRY / "covid_regions.gpkg").to_string(index=False))
    w = gpd.read_file(GEOMETRY / "covid_regions.gpkg", layer="region_w")
    print()
    print("region_w: {} features, CRS {}, {:,} cases".format(
        len(w), w.crs, int(w.TotalConfirmedCases.sum())))
except ImportError:
    print("geopandas not installed - see environment.yml.")
    print("Nothing above this cell needed it.")

     name geometry_type
region_ne  MultiPolygon
 region_n  MultiPolygon
region_nw  MultiPolygon
 region_w  MultiPolygon
region_sw  MultiPolygon
 region_s  MultiPolygon
region_se  MultiPolygon
 region_e  MultiPolygon

region_w: 22 features, CRS EPSG:4269, 10,341 cases


## 7. How to cite

`CITATION.cff` in the repo root gives GitHub's "Cite this repository" button. Cite the paper for the
work and the repository for the code — see the README.
